In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    f1_score
)

In [ ]:
from google.colab import files


print("Please upload credit_card_transactions.csv")
uploaded = files.upload()
filename = next(iter(uploaded))

# Load into DataFrame
df = pd.read_csv(filename)

print("Shape:", df.shape)
print("\nColumn info:")
df.info()

display(df.head())

In [ ]:
fraud_counts = df["IsFraud"].value_counts()
fraud_pct = df["IsFraud"].value_counts(normalize=True) * 100

print("Class counts:\n", fraud_counts)
print("\nClass percentage:\n", fraud_pct.round(2))

display(pd.DataFrame({"Counts": fraud_counts, "Percentage (%)": fraud_pct.round(2)}))

In [ ]:
print("Overall statistics:")
display(df.describe().round(2))

print("\nAverage values, Normal vs Fraud:")
display(df.groupby("IsFraud").mean(numeric_only=True).round(2))

In [ ]:
feature_cols = [
    "TransactionAmount",
    "TransactionHour",
    "CustomerAge",
    "AccountAgeDays",
    "NumPrevTransactions",
    "DistanceFromHomeKM"
]

X = df[feature_cols]
y = df["IsFraud"]
transaction_ids = df["TransactionID"]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

In [ ]:
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, transaction_ids,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train), " | Fraud in train:", y_train.sum())
print("Testing samples :", len(X_test), " | Fraud in test :", y_test.sum())

display(X_train.head(3))

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", y_train_sm.value_counts())

display(pd.DataFrame({"Before SMOTE": y_train.value_counts(), "After SMOTE": y_train_sm.value_counts()}))

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_sm, y_train_sm)
print("Model trained successfully.")

In [ ]:
y_pred_default = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]   # probability of class 1 (fraud)

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred_default))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_default, target_names=["Normal", "Fraud"]))

print("ROC-AUC score:", round(roc_auc_score(y_test, y_proba), 3))

In [ ]:
thresholds = np.arange(0.1, 0.95, 0.05)
results = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    report = classification_report(y_test, y_pred_t, output_dict=True, zero_division=0)
    results.append({
        "Threshold": round(t, 2),
        "Precision": round(report["1"]["precision"], 3),
        "Recall": round(report["1"]["recall"], 3),
        "F1-score": round(f1, 3)
    })

results_df = pd.DataFrame(results)
best_threshold = 0.25

display(results_df)

In [ ]:
y_pred_tuned = (y_proba >= best_threshold).astype(int)

print(f"Results at tuned threshold = {best_threshold}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=["Normal", "Fraud"]))

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

print("Feature importance scores:")
display(importances.to_frame(name="Importance Score").round(3))

plt.figure(figsize=(7, 4))
importances.sort_values().plot(kind="barh")
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance score")
plt.tight_layout()
plt.show()

In [ ]:
submission = pd.DataFrame({
    "TransactionID": id_test.values,
    "FraudProbability": np.round(y_proba, 4),
    "PredictedIsFraud": y_pred_tuned
})

submission.to_csv("submission.csv", index=False)
files.download("submission.csv")
display(submission.head(10))

In [ ]:
summary = pd.DataFrame([
    {
        "Setting": "Default threshold (0.5)",
        "Precision": round(classification_report(y_test, y_pred_default, output_dict=True, zero_division=0)["1"]["precision"], 3),
        "Recall": round(classification_report(y_test, y_pred_default, output_dict=True, zero_division=0)["1"]["recall"], 3),
        "F1-score": round(f1_score(y_test, y_pred_default), 3)
    },
    {
        "Setting": f"Tuned threshold ({best_threshold})",
        "Precision": round(classification_report(y_test, y_pred_tuned, output_dict=True, zero_division=0)["1"]["precision"], 3),
        "Recall": round(classification_report(y_test, y_pred_tuned, output_dict=True, zero_division=0)["1"]["recall"], 3),
        "F1-score": round(f1_score(y_test, y_pred_tuned), 3)
    }
])

display(summary)